## Register High-res H&E, Cy-IF & DESI WSI series W/ VALIS

In [ ]:
import os
import sys
import gc

import gcsfs
import tifffile
import numpy as np
import pandas as pd
import seaborn as sns
from skimage import io


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys
sys.path.append('..')
sys.path.append('../util/')
import registration, IO, utils

### H&E alignment

#### Data loading & alignment

In [ ]:
def disp_chans(img, title=None, ncols=4):
    depth = len(img)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(img[idx])
            idx += 1
            
    fig.tight_layout()
    fig.suptitle(title, y=1.01)
    plt.show()

In [ ]:
data_path = '../data/desi/post_desi_HE/'
res_path = '../data/desi/post_desi_HE_aligned/'
ref_slide = 'HE_slice_48.tif'

filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif']

imgs = [tifffile.imread(os.path.join(data_path, f))
        for f in filenames]


In [ ]:
registrar, _, _ = registration.run_valis(src_dir=data_path,
                                         res_dir=res_path,
                                         ref_slide=ref_slide,
                                         micro=False,
                                         kill_jvm=True)

In [ ]:
# Generate overlayed H&E WSI series
aligned_imgs = [tifffile.imread(os.path.join(res_path, 'registered_slides', f))
                for f in sorted(os.listdir(os.path.join(res_path, 'registered_slides')))
                if f[-8:] == 'ome.tiff']

aligned_imgs = np.array(aligned_imgs)
aligned_imgs = aligned_imgs.transpose((3,0,1,2))
tifffile.imwrite(os.path.join(res_path, 'valis_stacked.ome.tif'), aligned_imgs, metadata={'axes': 'CZYX'})

del aligned_imgs

#### Evaluation

- Comparison of registration results btw `rigid`, `non-rigid` & `micro-registration`

In [ ]:
# Per-slice errors before & after alignment
summary_df = pd.read_csv(os.path.join(res_path, 'data/_summary.csv'), index_col=[0])
summary_df.head()

In [ ]:
# rTRE score: relative Target Registration Error
# (avg. normed. dist btw landmarks from "src" to "dst")

nslides = summary_df.shape[0]

rTRE_df = pd.DataFrame({
    'name': nslides*['orig'] + nslides*['rigid'] + nslides*['non-rigid'],
    'rTRE': summary_df['original_rTRE'].to_list() + summary_df['rigid_rTRE'].to_list() + summary_df['non_rigid_rTRE'].to_list()
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x='name', y='rTRE', data=rTRE_df,
               palette=['cyan','g','orange'], ax=ax)
ax.set_title('relative Target Registration Error')
plt.show()

In [ ]:
nr_margin_df = pd.DataFrame({
    'name': nslides*['rigid'] + nslides*['non-rigid'],
    'rTRE': summary_df['rigid_rTRE'].to_list() + summary_df['non_rigid_rTRE'].to_list()
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x='name', y='rTRE', data=nr_margin_df,
            palette=['g','orange'], ax=ax)

ax.set_title('Rigid vs. Non-rigid')
plt.show()

Both non-rigid & micro-registration introduces artifacts outside the registered bounding-box (likely introduced from VALIS's pyramid schemes); Trim off before using the images)

---

### DESI alignment

Alignment of pos-mode (target) & neg-mode (source) of 2D paired DESI samples

In [ ]:
from importlib import reload
reload(registration)
reload(utils)

In [ ]:
%ls ../data/desi/desi_2d/pos/

In [ ]:
desi_pos_path = '../data/desi/desi_2d/pos/'
desi_neg_path = '../data/desi/desi_2d/neg/'

# TMP: redo sample 8 & 9, manually mirror...

# Load pos & neg mode images, normalize each channel to [0, 1]
filenames = sorted([f for f in os.listdir(desi_pos_path) if f[-3:] == 'tif'])[-2:]
pos_imgs = [utils.norm_by_channel(tifffile.imread(os.path.join(desi_pos_path, f)))
            for f in filenames][-2:]
neg_imgs = [utils.norm_by_channel(tifffile.imread(os.path.join(desi_neg_path, f)))
            for f in filenames][-2:]

# Load pos & neg m/z annotations
sample_filename = os.path.join(desi_pos_path, filenames[0])
with open(sample_filename, 'rb') as ifile:
    pos_labels = IO.load_ome_labels(ifile, sample_filename)

sample_filename = os.path.join(desi_neg_path, filenames[0])
with open(sample_filename, 'rb') as ifile:
    neg_labels = IO.load_ome_labels(ifile, sample_filename)

assert len(pos_labels) == len(pos_imgs[0]) and len(neg_labels) == len(neg_imgs[0]), \
    "Inconsistent btw m/z annotation & feature lengths"
del sample_filename

**Interactive anchor point annotations**

---

In [ ]:
%matplotlib widget

Annotate reference & target points

In [ ]:
pos_anchor_points = []
neg_anchor_points = []

pos_idx = 0
neg_idx = 0

In [ ]:
points = []
def onclick(event):
    ix, iy = event.xdata, event.ydata
    if ix is not None and iy is not None:
        points.append((ix, iy))
        print(f"Clicked pixel: x={ix:.2f}, y={iy:.2f}")

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(pos_imgs[pos_idx].mean(0), cmap='magma')   # show 2D Average Intensity Projection (AIP) across channels 
plt.show()
cid = fig.canvas.mpl_connect('button_press_event', onclick)


In [ ]:
pos_anchor_points.append(points)
print("{0} pts; {1}/{2} images annotated so far".format(len(points), len(pos_anchor_points), len(pos_imgs)))
pos_idx += 1
del points

In [ ]:
points = []
def onclick(event):
    ix, iy = event.xdata, event.ydata
    if ix is not None and iy is not None:
        points.append((ix, iy))
        print(f"Clicked pixel: x={ix:.2f}, y={iy:.2f}")


fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(neg_imgs[neg_idx].mean(0), cmap='magma')   # show 2D Average Intensity Projection (AIP) across channels 
plt.show()
cid = fig.canvas.mpl_connect('button_press_event', onclick)


In [ ]:
neg_anchor_points.append(points)
print("{0} pts; {1}/{2} images annotated so far".format(len(points), len(neg_anchor_points), len(neg_imgs)))
neg_idx += 1
del points

---

In [ ]:
%matplotlib inline

#### Rigid Alignment

Neg --> Pos

In [ ]:
neg_imgs_warped = []
Ms = []

for pos_img, neg_img, pos_pts, neg_pts in zip(pos_imgs, neg_imgs, pos_anchor_points, neg_anchor_points):

    # (1). obtain affine transformation for a single channel btw `+` & `-` AIP image
    M = registration.get_affine_matrix(source=neg_img.mean(0),  # dim: [Y', X']
                                       target=pos_img.mean(0),  # dim: [Y, X]
                                       pts_source=neg_pts,
                                       pts_target=pos_pts)
    Ms.append(M)

    # (2). affine transformation across all `-` mode channels
    nchan = neg_img.shape[0]
    neg_img_warped = np.zeros((nchan,) + pos_img.shape[1:], dtype=np.float32)
    for i in range(nchan):
        neg_img_warped[i] = registration.affine_warp(neg_img[i], 
                                                     pos_img.shape[1:],  # dim: [Y, X]
                                                     M)
    neg_imgs_warped.append(neg_img_warped)

del pos_img, neg_img, neg_img_warped, pos_pts, neg_pts, M

8 & 9 doesn't work

#### Non-rigid Alignment

In [ ]:
bk_dxdys = []
neg_imgs_nr_warped = []

for pos_img, neg_img in zip(pos_imgs, neg_imgs_warped):

    # (1). obtain optical flow for a single channel btw `+` & `-` images
    _, bk_dxdy = registration.non_rigid_warp(neg_img.mean(0), pos_img.mean(0))
    bk_dxdys.append(bk_dxdy)

    # (2). Non-rigid transformation across all channels
    neg_img_warped = np.zeros_like(neg_img)
    for i, chan in enumerate(neg_img):
        neg_img_warped[i] = registration.non_rigid_warp(chan, pos_img.mean(0), bk_dxdy=bk_dxdy)[0]
    neg_imgs_nr_warped.append(neg_img_warped)

del pos_img, neg_img, neg_img_warped, bk_dxdy

Stack positive & negative channels

In [ ]:
merged_imgs = [np.vstack((pos_img, neg_img)) for (pos_img, neg_img) in zip(pos_imgs, neg_imgs_nr_warped)]
labels = pos_labels + neg_labels

out_path = '../data/desi/desi_2d/merged/'
#if not os.path.exists(out_path):
#    os.makedirs(out_path, exist_ok=True)

for filename, img in zip(filenames, merged_imgs):
    print('Saving +- merged {}...'.format(filename))
    tifffile.imwrite(
        os.path.join(out_path, filename),
        img,
        metadata={
            'axes': 'CYX',
            'Channel': {'Name': labels}
        }
    )

### Post-modality H&E alignment

Alignment of post-DESI H&E slices

In [ ]:
from importlib import reload
reload(registration)
reload(utils)

In [ ]:
def disp_chans(img, title=None, ncols=4):
    depth = len(img)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(img[idx])
            idx += 1
            
    fig.tight_layout()
    fig.suptitle(title, y=1.01)
    plt.show()


In [ ]:
# Load post-H&E slices from gcloud
# CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
# PROJECT_ID = 'azizilab-cell-segmentation'
# BUCKET_ID = 'liver_3d'
# HOME_PATH = 'CyIF/H&E_post_CyIF'

# he_reader = IO.GcloudReader(CREDENTIAL_PATH,
#                             PROJECT_ID,
#                             BUCKET_ID,
#                             HOME_PATH)


# he_imgs = he_reader.load_imgs()

# Load H&E post-H&E slices locally
data_path = '../data/desi/post_desi_HE/'
res_path = '../data/desi/post_desi_HE_aligned/'
ref_slide = 'HE_slice_48.tif'

filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif']

he_imgs = [tifffile.imread(os.path.join(data_path, f))
        for f in filenames]

print(len(he_imgs))

In [ ]:
disp_chans(he_imgs)

#### Rigid Alignment

##### Interactive anchor point annotations

---

In [ ]:
%matplotlib widget

In [ ]:
idx = 0
anchor_points = []

In [ ]:
points = []
def onclick(event):
    ix, iy = event.xdata, event.ydata
    if ix is not None and iy is not None:
        points.append((ix, iy))
        print(f"Clicked pixel: x={ix:.2f}, y={iy:.2f}")


fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(he_imgs[idx])
plt.show()
cid = fig.canvas.mpl_connect('button_press_event', onclick)

In [ ]:
len(points)

In [ ]:
anchor_points.append(points)
print(len(anchor_points), "points annotated so far")
idx += 1
del points

In [ ]:
# Save self-annotated anchor points

anchor_path = '../data/desi/post_desi_anchors/'
if not os.path.exists(anchor_path):
    os.makedirs(anchor_path, exist_ok=True)

for filename, pts in zip(filenames, anchor_points):
    np.savetxt(os.path.join(anchor_path, filename.split('.')[0]+'.pts'), pts)

---

In [ ]:
%matplotlib inline

In [ ]:
# Load self-annotated anchor points
data_path = '../data/desi/post_desi_anchors/'
anchor_points = IO.load_anchor_points(data_path)

In [ ]:
ref_idx = 6
ref_name = filenames[ref_idx]
ref_img = he_imgs[ref_idx]
ref_pts = anchor_points[ref_idx]

M_dict = {}

for src_name, src_pts, img in zip(filenames, anchor_points, he_imgs):

    if src_name == ref_name:
        continue

    print('Computing M for {}...'.format(src_name))
    
    M = registration.get_affine_matrix(source=img,
                                       target=ref_img,
                                       pts_source=src_pts,
                                       pts_target=ref_pts
                                      )
    M_dict[src_name] = M

    del M, img, src_name
    gc.collect()

del src_pts, ref_pts

In [ ]:
warped_imgs = np.zeros((len(he_imgs),) + ref_img.shape, dtype=np.uint8)

for i, filename in enumerate(filenames):
    if filename == ref_name:
        warped_imgs[i] = ref_img
    else:
        print('Warping image {}...'.format(filename))
        warped_imgs[i] = registration.affine_warp(he_imgs[i], ref_img.shape, M_dict[filename])


In [ ]:
# Save Affine-warped images
out_path = '../data/desi/post_desi_HE_rigid/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

for filename, img in zip(filenames, warped_imgs):
    tifffile.imwrite(os.path.join(out_path, filename), img)

del out_path, warped_imgs
gc.collect()

#### Non-rigid Alignment

In [ ]:
# Load pre-aligned images w/ Affine transformation
data_path = '../data/desi/post_desi_HE_rigid/'
filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif']
he_imgs = [tifffile.imread(os.path.join(data_path, f))
           for f in filenames]

In [ ]:
# WSI non-rigid alignment & save warped images

out_path = '../data/desi/post_desi_HE_non_rigid/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

ref_idx = 6
ref_name = filenames[ref_idx]
ref_img = he_imgs[ref_idx]
non_rigid_warped_imgs = np.zeros((len(he_imgs),) + ref_img.shape, dtype=np.uint8)

for i, (filename, src_img) in enumerate(zip(filenames, he_imgs)):
    print('Non-rigid Alignment + Warping on {}...'.format(filename))
    if filename == ref_name:
        warped_img = ref_img
    else:
        # dim: [C, Y, X]
        warped_img, _ = registration.non_rigid_warp(src_img, ref_img) 
    non_rigid_warped_imgs[i] = warped_img
    tifffile.imwrite(os.path.join(out_path, filename), warped_img)


In [ ]:
# Visualization
aln_imgs = [tifffile.imread(os.path.join(out_path, filename))
            for filename in filenames]
disp_chans(aln_imgs)

In [ ]:
# Save final aligned images with FOV
# r = 1300
# ymid, xmid = non_rigid_warped_imgs[6].shape[0]//2+100, non_rigid_warped_imgs[0].shape[1]//2+100
# out_path = '../data/desi/post_desi_HE_aligned/'
# if not os.path.exists(out_path):
#     os.makedirs(out_path, exist_ok=True)

# for filename, img in zip(filenames, non_rigid_warped_imgs):
#     trimmed_img = img[ymid-r:ymid+r, xmid-r:xmid+r]
#     tifffile.imwrite(os.path.join(out_path, filename), trimmed_img)

# del filename, img, trimmed_img

In [ ]:
# Save WSI image
tifffile.imwrite(os.path.join(out_path, 'HE_series.ome.tif'),
                 non_rigid_warped_imgs.transpose(3,0,1,2),  # Dim: [C, Z, Y, X]
                 metadata={'axes': 'CZYX'})

In [ ]:
# Sample Non-rigid alignment & visualization

# sample_moving_img = he_imgs[0]
# sample_fixed_img = he_imgs[18]
# sample_warped_img, bk_dxdy = IO.non_rigid_warp(sample_moving_img, sample_fixed_img)

# fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(1, 5, figsize=(10, 2.2))
# ax1.imshow(sample_moving_img)
# ax1.axis('off')
# ax1.set_title('Source')
# ax2.imshow(sample_fixed_img)
# ax2.axis('off')
# ax2.set_title('Target')
# ax3.imshow(backward_flow[0], cmap='coolwarm')
# ax3.axis('off')
# ax3.set_title('Optical Flow (dX)')
# ax4.imshow(backward_flow[1], cmap='coolwarm')
# ax4.axis('off')
# ax4.set_title('Optical Flow (dY)')
# ax5.imshow(sample_warped_img)
# ax5.axis('off')
# ax5.set_title('Source (warped)')

# plt.tight_layout()
# plt.show()

### Post-modality multi-modal alignment

In [ ]:
import shutil
# from importlib import reload
# reload(registration)

- Align CyCIF to post-DESI H&E:

Create parallel structure
```
src_dir/
 ├─ slice_01/
 │  ├─ he_image.tif
 │  ├─ cyif_image.ome.tif
 │  ├─ ...
 ├─ slice_02/
 ├─ slice_03/
 ├─ ...
```

In [ ]:
pre_align_path = '../data/desi/he_cycif/'  # Axial-one-to-one mapped ref & target images
post_align_path = '../data/desi/post_desi_cycif_aligned/'  # Cross-modality alignment output directory
he_path = '../data/desi/post_desi_HE_aligned/'
cyif_path = '../data/desi/post_desi_cycif/'

In [ ]:
def create_multialn_files(src_dir, ref_dir, out_dir, 
                          subdir_prefix='slice_'):
    """Create parallel structure for multi-modal VALIS alignment"""
    assert os.path.exists(src_dir) and os.path.exists(ref_dir)
    src_filenames = [f for f in sorted(os.listdir(src_dir)) if f[-3:] == 'tif']
    ref_filenames = [f for f in sorted(os.listdir(ref_dir)) if f[-3:] == 'tif']
    assert len(src_filenames) == len(ref_filenames), "Requires equal # images for each modality"
    
    if not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)
        
    subdirs = [subdir_prefix+str(idx) if idx >= 10 
               else subdir_prefix+'0'+str(idx)
               for idx in range(len(src_filenames))]

    for sf, rf, subdir in zip(src_filenames, ref_filenames, subdirs):
        if not os.path.exists(os.path.join(out_dir, subdir)):
            os.makedirs(os.path.join(out_dir, subdir), exist_ok=True)
        shutil.copy(os.path.join(src_dir, sf), os.path.join(out_dir, subdir))
        shutil.copy(os.path.join(ref_dir, rf), os.path.join(out_dir, subdir))
        
    return None


In [ ]:
create_multialn_files(src_dir=cyif_path,
                      ref_dir=he_path,
                      out_dir=pre_align_path)

registration.run_valis_multi(src_dir=pre_align_path,
                             res_dir=post_align_path,
                             ref_prefix='HE_',
                             kill_jvm=True)

In [ ]:
%ls ../data/desi/he_cycif_aligned/registered_slides/

In [ ]:
# Test alignment
# cyif_aligned = [tifffile.imread(os.path.join('../data/desi/he_cycif_aligned/registered_slides/', f))
#                 for f in sorted(os.listdir('../data/desi/he_cycif_aligned/registered_slides/'))
#                 if f[-3:] == 'tif']

- Align DESI to post-DESI H&E: <br>Align each DESI image to the backbone, **downscaled, grayscale** H&E via landmark annotations + Non-rigid refining

In [ ]:
from skimage.color import rgb2gray
from skimage.transform import rescale

In [ ]:
he_path = '../data/desi/post_desi_HE_aligned/'

# Load aligned HE images, transform to grayscale & downscale
scale_ratio = 1/10
he_imgs = [tifffile.imread(os.path.join(he_path, f))
           for f in sorted(os.listdir(he_path))
           if f[-3:] == 'tif']


he_imgs_grayscale = [rescale(rgb2gray(img), scale=scale_ratio, preserve_range=True).astype(np.float32)
                     for img in he_imgs]
del he_imgs

desi_path = '../data/desi/neg_ion/'
desi_imgs = [tifffile.imread(os.path.join(desi_path, f))
             for f in sorted(os.listdir(desi_path))
             if f[-3:] == 'tif']
desi_imgs = [utils.norm_by_channel(img) for img in desi_imgs]


##### Interactive anchor point annotations

---

In [ ]:
%matplotlib widget

Annotate reference points:

In [ ]:
he_anchor_points = []
idx = 0

In [ ]:
points = []
def onclick(event):
    ix, iy = event.xdata, event.ydata
    if ix is not None and iy is not None:
        points.append((ix, iy))
        print(f"Clicked pixel: x={ix:.2f}, y={iy:.2f}")


fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(he_imgs_grayscale[idx])
plt.show()
cid = fig.canvas.mpl_connect('button_press_event', onclick)

In [ ]:
he_anchor_points.append(points)
print(len(he_anchor_points), "points annotated so far")
idx += 1
del points

In [ ]:
desi_anchor_points = []
idx = 0

Annotate source points

In [ ]:
points = []
def onclick(event):
    ix, iy = event.xdata, event.ydata
    if ix is not None and iy is not None:
        points.append((ix, iy))
        print(f"Clicked pixel: x={ix:.2f}, y={iy:.2f}")


fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(desi_imgs[idx][10])
plt.show()
cid = fig.canvas.mpl_connect('button_press_event', onclick)

In [ ]:
desi_anchor_points.append(points)
print(len(desi_anchor_points), "points annotated so far")
idx += 1
del points

---

In [ ]:
%matplotlib inline

In [ ]:
# Rigid alignment: individual DESI (src) chan to grayscale H&E (ref)
desi_rigid_warped = []
Ms = []

for desi_img, he_img, desi_pts, he_pts in zip(desi_imgs, he_imgs_grayscale, desi_anchor_points, he_anchor_points):

    # (1). obtain affine transformation btw a single DESI channel and HE reference
    M = registration.get_affine_matrix(source=desi_img[10],
                                       target=he_img,
                                       pts_source=desi_pts,
                                       pts_target=he_pts)
    Ms.append(M)
    
    # (2). affine transformation across all DESI channels
    nchan = desi_img.shape[0]
    desi_warped = np.zeros((nchan,) + he_img.shape, dtype=np.float32)
    for i in range(nchan):
        desi_warped[i] = registration.affine_warp(desi_img[i], he_img.shape, M)
    desi_rigid_warped.append(desi_warped)

del desi_img, he_img, desi_warped, desi_pts, he_pts, M

- Non-rigid Alignment

In [ ]:
bk_dxdys = []
desi_nonrigid_warped = []

for desi_img, he_img in zip(desi_rigid_warped, he_imgs_grayscale):

    # (1). obtain optical flow btw a single DESI channel and HE reference
    _, bk_dxdy = registration.non_rigid_warp(desi_img[0], he_img)
    bk_dxdys.append(bk_dxdy)

    # (2). Non-rigid transformation across al DESI channels
    desi_warped = np.zeros_like(desi_img)
    for i, chan in enumerate(desi_img):
        desi_warped[i] = registration.non_rigid_warp(chan, he_img, bk_dxdy=bk_dxdy)[0]
    desi_nonrigid_warped.append(desi_warped)

del desi_img, he_img, desi_warped, bk_dxdy

In [ ]:
# Save images (note: metabolite labels X available, append later)
out_path = '../data/desi/desi_aligned'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)
desi_filenames = [f for f in sorted(os.listdir('../data/desi/raw/')) if f[-3:] == 'tif']

for filename, img in zip(desi_filenames, desi_nonrigid_warped):
    tifffile.imwrite(os.path.join(out_path, filename), img)

In [ ]:
# tmp: attach missing annotations & remove duplicated channel
ifile = open('../data/integrated/DESI_tmp.ome.tif', 'rb')
ion_labels = IO.load_ome_labels(ifile, '../data/integrated/DESI_tmp.ome.tif')
ifile.close()

In [ ]:
ion_labels[174]

In [ ]:
# Index of duplicated label: 85, 174
# remove idx 174 from both annot & original img channel
ion_labels = np.concatenate((np.array(ion_labels)[:174], 
                             np.array(ion_labels)[175:]))

In [ ]:
desi_path = '../data/desi/desi_aligned/'
desi_filenames = [f for f in sorted(os.listdir(desi_path)) if f[-3:] == 'tif']
desi_dup = [tifffile.imread(os.path.join(desi_path, f)) for f in desi_filenames]

desi_raw = [np.concatenate((desi_dup[i][:174], desi_dup[i][175:]))
            for i in range(len(desi_dup))]

In [ ]:
annot_imgs = {
    filename: {
        ion_label: chan
        for (ion_label, chan) in zip(ion_labels, img)
    }
    for (filename, img) in zip(desi_filenames, desi_raw)
}

In [ ]:
IO.save_annot_tiffs(annot_imgs, path='../data/desi/desi_aligned_new/')

Create ROI patches:

In [ ]:
he_path = '../data/desi/post_desi_HE_aligned/'
he_filenames = [os.path.join(he_path, f) for f in sorted(os.listdir(he_path)) if f[-3:] == 'tif']
he_aligned = [tifffile.imread(f) for f in he_filenames]

In [ ]:
cyif_path = '../data/desi/he_cycif_aligned/registered_slides/'
cyif_filenames = [os.path.join(cyif_path, f) for f in sorted(os.listdir(cyif_path)) if f[-3:] == 'tif']
cyif_aligned = [tifffile.imread(f) for f in cyif_filenames]

In [ ]:
desi_path = '../data/desi/desi_aligned/'
desi_filenames = [os.path.join(desi_path, f) for f in sorted(os.listdir(desi_path)) if f[-3:] == 'tif']
desi_aligned = [tifffile.imread(f) for f in desi_filenames]

---